# OccuMap — LLM Occupation Classifier

Personal recreation of an LLM-based occupation classifier using Python and the Claude API.

Maps messy CRM occupation titles to Singapore SSOC 2024 — outputting PME / T / RNF / NA labels
alongside mapped SSOC codes, confidence scores, and a human-in-the-loop review workflow.

**Pipeline:**
1. Rule-based pre-processor — fast-path obvious cases
2. LLM classifier — Claude API with 6 consistency mechanisms  
3. Human review queue — Streamlit app for low-confidence cases
4. Final labels — merged output for Tableau dashboard

See METHODOLOGY.md for full documentation.

In [52]:
import pandas as pd
import anthropic
import json
import os
from dotenv import load_dotenv
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Load API key from .env
load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

print("Setup complete")

Setup complete


In [53]:
# ── Global Configuration ──────────────────────────────────────────────────────
MODEL = "claude-sonnet-4-6"
PROMPT_VERSION = "v1.3"
CONFIDENCE_THRESHOLD = 0.75
MAJORITY_VOTE_RUNS = 3

print(f"Model:               {MODEL}")
print(f"Prompt version:      {PROMPT_VERSION}")
print(f"Confidence threshold:{CONFIDENCE_THRESHOLD}")
print(f"Majority vote runs:  {MAJORITY_VOTE_RUNS}")

Model:               claude-sonnet-4-6
Prompt version:      v1.3
Confidence threshold:0.75
Majority vote runs:  3


In [54]:
# Load synthetic occupation titles
df = pd.read_csv("data/synthetic_titles.csv")

print(f"Total records: {len(df)}")
print(f"\nFirst 10 titles:")
print(df.head(10))
print(f"\nSample of short/messy entries:")
print(df[df['CRM_OCCUPATION_TITLE'].str.len() < 5].head(10))

Total records: 297

First 10 titles:
       CRM_OCCUPATION_TITLE
0               SR MKTG MGR
1                         -
2                      CHEF
3  CHIEF TECHNOLOGY OFFICER
4     MECHANICAL TECHNICIAN
5                 ARCHITECT
6            SECURITY GUARD
7                BUS DRIVER
8            GENERAL WORKER
9                   RETIRED

Sample of short/messy entries:
    CRM_OCCUPATION_TITLE
1                      -
2                   CHEF
68                  TECH
80                   NIL
87                   CSE
186                 COOK


In [55]:
import re

# ── Rule-based pre-processor ──────────────────────────────────────────────────
# Stage 1 of pipeline — fast-path obvious cases before LLM

NA_KEYWORDS = [
    'resign', 'retrenched', 'retrench', 'retired', 'retiree', 'retirement',
    'housewife', 'homemaker', 'home maker', 'unemployed',
    'nsf', 'ns man', 'full-time ns', 'landlord',
    'nil', 'none', 'n/a', 'not applicable', 'unknown', 'unkown',
    'others', 'retired pensioner', 'pensioner',
    'not ', 'resigned'
]

RNF_DRIVER_KEYWORDS = [
    'taxi driver', 'grab driver', 'uber driver', 'private hire driver',
    'private hired driver', 'phv driver', 'bus driver', 'bus captain',
    'tipper truck driver', 'truck driver', 'dispatch rider', 'delivery rider',
    'food delivery rider', 'crawler crane driver', 'dedicated truck driver',
    'private driver'
]

RNF_OTHER_KEYWORDS = [
    'hawker', 'stall holder', 'stall vendor', 'confinement lady'
]

PME_KEYWORDS = [
    'pastor', 'missionary', 'businessman', 'business owner', 'pub owner'
]

LLM_OVERRIDE_KEYWORDS = [
    'freelance', 'freelancer', 'self employ', 'self-employ',
    'contractor', 'church worker', 'canteen operator'
]

GIBBERISH_PATTERNS = [
    r'^-+$',
    r'^\d+$',
    r'^[a-zA-Z]{1,2}$',
    r'^left \d+',
    r'^bc - \d+',
    r'^pt bus captain - \d+',
    r'^bus captain - \d+',
    r'^snr? [a-z] [a-z]$',
]

def preprocess(title):
    if pd.isna(title) or str(title).strip() == '':
        return title, 'NA'

    clean = str(title).strip()
    clean = re.sub(r'\s+', ' ', clean)
    clean = re.sub(r'^\.\.\.', '', clean)
    clean = clean.strip()
    lower = clean.lower()

    for pattern in GIBBERISH_PATTERNS:
        if re.match(pattern, lower):
            return clean, 'NA'

    if len(clean) <= 2:
        return clean, 'NA'

    for keyword in LLM_OVERRIDE_KEYWORDS:
        if keyword in lower:
            return clean, None

    for keyword in NA_KEYWORDS:
        if keyword in lower:
            return clean, 'NA'

    for keyword in PME_KEYWORDS:
        if keyword in lower:
            return clean, 'PME'

    for keyword in RNF_DRIVER_KEYWORDS:
        if keyword in lower:
            return clean, 'RNF'

    for keyword in RNF_OTHER_KEYWORDS:
        if keyword in lower:
            return clean, 'RNF'

    return clean, None


# Apply pre-processor
df['cleaned_title'] = df['CRM_OCCUPATION_TITLE'].apply(lambda x: preprocess(x)[0])
df['fast_path'] = df['CRM_OCCUPATION_TITLE'].apply(lambda x: preprocess(x)[1])

total = len(df)
print(f"Total records:   {total}")
print(f"Fast-path → NA:  {(df['fast_path'] == 'NA').sum()}")
print(f"Fast-path → RNF: {(df['fast_path'] == 'RNF').sum()}")
print(f"Fast-path → PME: {(df['fast_path'] == 'PME').sum()}")
print(f"Needs LLM:       {df['fast_path'].isna().sum()}")

Total records:   297
Fast-path → NA:  8
Fast-path → RNF: 10
Fast-path → PME: 2
Needs LLM:       277


In [56]:
# Load SSOC 2024 Detailed Definitions
ssoc_df = pd.read_excel(
    "data/ssoc2024-detailed-definitions.xlsx",
    sheet_name="SSOC2024 Detailed Definitions",
    header=4
)

ssoc_df.columns = [
    "ssoc_code", "title", "groups", "definition",
    "tasks", "notes", "examples_here", "examples_elsewhere"
]

ssoc_df = ssoc_df[ssoc_df["ssoc_code"].notna()].copy()
ssoc_df["ssoc_code"] = ssoc_df["ssoc_code"].astype(str).str.strip()
ssoc_df["major_group"] = ssoc_df["ssoc_code"].str[0]

def get_split(major_group):
    if major_group in ["1", "2"]:
        return "PME"
    elif major_group == "3":
        return "T"
    elif major_group in ["4", "5", "6", "7", "8", "9"]:
        return "RNF"
    else:
        return "NA"

ssoc_df["split"] = ssoc_df["major_group"].apply(get_split)
ssoc_5digit = ssoc_df[ssoc_df["ssoc_code"].str.len() == 5].copy()

print(f"Total SSOC rows:          {len(ssoc_df)}")
print(f"5-digit occupation codes: {len(ssoc_5digit)}")

Total SSOC rows:          1617
5-digit occupation codes: 1006


/Users/sebastianlim/Library/CloudStorage/Dropbox/NTUC Employment 2024/PulseProject/occumap/.venv/lib/python3.13/site-packages/openpyxl/reader/workbook.py:118: UserWarning: Print area cannot be set to Defined name: 'SSOC2024 Detailed Definitions'!$A:$F.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")
/Users/sebastianlim/Library/CloudStorage/Dropbox/NTUC Employment 2024/PulseProject/occumap/.venv/lib/python3.13/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [57]:
# ── Static major group context ────────────────────────────────────────────────
major_group_context = """
SSOC 2024 Major Groups and Classification:

PME (Professional, Manager, Executive):
- Group 1: Legislators, Senior Officials and Managers
  Includes: CEOs, directors, general managers, department heads,
  senior government officials, politicians
- Group 2: Professionals
  Includes: Engineers, doctors, lawyers, accountants, architects,
  scientists, teachers, nurses, social workers, analysts, IT professionals,
  pilots, pharmacists, designers (professional)

T (Technician / Associate Professional):
- Group 3: Associate Professionals and Technicians
  Includes: Engineering technicians, medical/lab technicians,
  IT support analysts, legal/business associate professionals,
  air traffic controllers, ship officers, police inspectors,
  pre-school educators, fitness instructors, photographers

RNF (Rank and File):
- Group 4: Clerical Support Workers
  Includes: Clerks, data entry operators, receptionists,
  secretaries, admin assistants, call centre agents
- Group 5: Services and Sales Workers
  Includes: Retail assistants, waiters, cooks, security officers,
  hairdressers, childcare workers, tour guides, cashiers
- Group 6: Agricultural and Fishery Workers
  Includes: Farm workers, fishermen
- Group 7: Craftsmen and Related Trades Workers
  Includes: Electricians, plumbers, welders, mechanics,
  carpenters, printers, tailors
- Group 8: Plant and Machine Operators and Assemblers
  Includes: Machine operators, drivers, crane operators,
  production/assembly workers
- Group 9: Cleaners, Labourers and Related Workers
  Includes: Cleaners, refuse collectors, general labourers,
  kitchen helpers, store hands

NA (Not Applicable):
- Group X: Workers not elsewhere classified
- Also use NA for: retired, resigned, unemployed, homemaker,
  NSF, blank, unknown, internal codes

Key classification rules:
- "Manager" in title → PME (Group 1) by default
- "Engineer" in title → PME (Group 2) by default
- "Technician" in title → T (Group 3) by default
- Drivers (taxi, bus, grab, private hire) → RNF (Group 8)
- "Assistant [occupation]" → follow the base occupation's group
- "Senior [occupation]" → follow the base occupation's group
- Self-employed titles → classify by the actual occupation
- Freelance [occupation] → classify by the actual occupation
"""

# ── Dynamic SSOC candidate lookup ────────────────────────────────────────────
def get_ssoc_candidates(title, top_n=15):
    if not title or str(title).strip() == '':
        return ""

    title_clean = str(title).lower().strip()
    title_words = set(title_clean.split())

    stopwords = {'and', 'or', 'the', 'of', 'in', 'at', 'to', 'for',
                 'a', 'an', 'senior', 'junior', 'assistant', 'associate',
                 'chief', 'head', 'lead', 'principal', 'deputy', 'asst',
                 'snr', 'sr', 'jnr', 'jr', 'pt', 'part', 'time'}

    content_words = title_words - stopwords
    if not content_words:
        return ""

    def score(row):
        ssoc_title = str(row['title']).lower()
        ssoc_words = set(ssoc_title.split()) - stopwords
        examples = str(row.get('examples_here', '')).lower()
        definition = str(row.get('definition', '')).lower()

        phrase_match = 10 if title_clean in ssoc_title else 0
        example_phrase = 8 if title_clean in examples else 0
        word_overlap = len(content_words & ssoc_words) * 2
        example_word = sum(2 for word in content_words if word in examples)
        definition_word = sum(1 for word in content_words if word in definition)

        return phrase_match + example_phrase + word_overlap + example_word + definition_word

    ssoc_5digit_copy = ssoc_5digit.copy()
    ssoc_5digit_copy['score'] = ssoc_5digit_copy.apply(score, axis=1)
    candidates = ssoc_5digit_copy[ssoc_5digit_copy['score'] > 0].nlargest(top_n, 'score')

    if candidates.empty:
        return ""

    lines = ["Candidate SSOC codes (choose from these if relevant):"]
    for _, row in candidates.iterrows():
        lines.append(f"- {row['ssoc_code']}: {row['title']} ({row['split']})")

    return "\n".join(lines)

print("SSOC context and candidate lookup ready.")

SSOC context and candidate lookup ready.


In [58]:
def single_api_call(title, prompt):
    """Single API call — used by parallel voter"""
    try:
        response = client.messages.create(
            model=MODEL,
            max_tokens=400,
            temperature=0,  # mechanism 1: deterministic
            messages=[{"role": "user", "content": prompt}]
        )
        raw = response.content[0].text.strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        return json.loads(raw)  # mechanism 2: structured JSON
    except Exception as e:
        return None


def classify_title(title, n_runs=MAJORITY_VOTE_RUNS):
    """
    Classify a single occupation title using Claude API.

    Consistency mechanisms:
    1. Temperature = 0 — deterministic output
    2. Structured JSON output — prevents format drift
    3. Confidence scoring + thresholding — auto-accept vs human review
    4. Majority voting — N parallel runs, take most common result
    5. Prompt versioning — every output tagged with prompt version
    6. Dynamic SSOC candidate injection — grounds LLM in real SSOC codes
    """
    candidates = get_ssoc_candidates(title)  # mechanism 6

    prompt = f"""You are classifying Singapore worker occupation titles using the SSOC 2024 (Singapore Standard Occupational Classification).

{major_group_context}

{candidates}

Classify this occupation title: "{title}"

Return ONLY a valid JSON object, nothing else. No explanation outside the JSON.
Use this exact format:
{{
  "classification": "PME",
  "major_group": "2",
  "major_group_title": "Professionals",
  "ssoc_code": "25121",
  "ssoc_occupation_title": "Software Developer",
  "confidence": 0.95,
  "reasoning": "Software Engineer falls under Group 2 Professionals"
}}

Rules:
- classification must be one of: PME, T, RNF, NA
- major_group must be one of: 1, 2, 3, 4, 5, 6, 7, 8, 9, X
- ssoc_code must be chosen from the candidate list above where possible
- ssoc_occupation_title must match the chosen ssoc_code exactly
- If classification is NA, set major_group to X, ssoc_code to X0000, ssoc_occupation_title to Not Applicable
- confidence must be a float between 0.0 and 1.0
- reasoning must be a single concise sentence"""

    # mechanism 4: majority voting via parallel calls
    results = []
    with ThreadPoolExecutor(max_workers=n_runs) as executor:
        futures = [executor.submit(single_api_call, title, prompt)
                   for _ in range(n_runs)]
        for future in as_completed(futures):
            result = future.result()
            if result:
                results.append(result)

    if not results:
        return {
            "classification": "NA",
            "major_group": "X",
            "major_group_title": "Unknown",
            "ssoc_code": "X0000",
            "ssoc_occupation_title": "Not Applicable",
            "confidence": 0.0,
            "reasoning": "Classification failed",
            "votes": 0,
            "review_status": "NEEDS_REVIEW",
            "prompt_version": PROMPT_VERSION
        }

    classifications = [r["classification"] for r in results]
    vote_counts = Counter(classifications)
    winning_label = vote_counts.most_common(1)[0][0]
    vote_count = vote_counts.most_common(1)[0][1]

    winning_results = [r for r in results if r["classification"] == winning_label]
    avg_confidence = sum(r["confidence"] for r in winning_results) / len(winning_results)
    best_result = max(winning_results, key=lambda r: r["confidence"])

    # mechanism 3: confidence thresholding
    if avg_confidence >= CONFIDENCE_THRESHOLD and vote_count >= 2:
        review_status = "AUTO_ACCEPTED"
    else:
        review_status = "NEEDS_REVIEW"

    return {
        "classification": winning_label,
        "major_group": best_result.get("major_group", "X"),
        "major_group_title": best_result.get("major_group_title", ""),
        "ssoc_code": best_result.get("ssoc_code", ""),
        "ssoc_occupation_title": best_result.get("ssoc_occupation_title", ""),
        "confidence": round(avg_confidence, 3),
        "reasoning": best_result.get("reasoning", ""),
        "votes": vote_count,
        "review_status": review_status,
        "prompt_version": PROMPT_VERSION  # mechanism 5
    }

print(f"classify_title {PROMPT_VERSION} loaded — with dynamic SSOC candidate injection")

classify_title v1.3 loaded — with dynamic SSOC candidate injection


In [59]:
# Quick sanity check on 10 representative titles
test_titles = [
    "SOFTWARE ENGINEER", "TAXI DRIVER", "GRAB DRIVER",
    "MANAGER", "CLEANING SUPERVISOR", "RETIRED PENSIONER",
    "BARISTA", "FINANCIAL ANALYST", "PROCESS TECHNICIAN",
    "FREELANCE PHOTOGRAPHER"
]

print("Testing classify_title on 10 titles...\n")
start = time.time()

for title in test_titles:
    t0 = time.time()
    result = classify_title(title)
    t1 = time.time()
    print(f"{title:<30} → {result['classification']:<4} | "
          f"SSOC: {result['ssoc_code']} | "
          f"Confidence: {result['confidence']:.2f} | "
          f"{result['review_status']} | {t1-t0:.1f}s")

print(f"\nTotal: {time.time()-start:.1f}s")

Testing classify_title on 10 titles...

SOFTWARE ENGINEER              → PME  | SSOC: 25121 | Confidence: 0.92 | AUTO_ACCEPTED | 2.5s
TAXI DRIVER                    → RNF  | SSOC: 83221 | Confidence: 0.99 | AUTO_ACCEPTED | 2.4s
GRAB DRIVER                    → RNF  | SSOC: 83226 | Confidence: 0.97 | AUTO_ACCEPTED | 2.8s
MANAGER                        → PME  | SSOC: 12112 | Confidence: 0.50 | NEEDS_REVIEW | 2.9s
CLEANING SUPERVISOR            → RNF  | SSOC: 91000 | Confidence: 0.95 | AUTO_ACCEPTED | 2.2s
RETIRED PENSIONER              → NA   | SSOC: X0000 | Confidence: 0.99 | AUTO_ACCEPTED | 2.5s
BARISTA                        → RNF  | SSOC: 51322 | Confidence: 0.99 | AUTO_ACCEPTED | 2.2s
FINANCIAL ANALYST              → PME  | SSOC: 24131 | Confidence: 0.99 | AUTO_ACCEPTED | 2.2s
PROCESS TECHNICIAN             → T    | SSOC: 31173 | Confidence: 0.85 | AUTO_ACCEPTED | 2.6s
FREELANCE PHOTOGRAPHER         → T    | SSOC: 34310 | Confidence: 0.95 | AUTO_ACCEPTED | 2.7s

Total: 25.0s


In [ ]:
# ── Full classification pipeline — batch parallel ─────────────────────────────
# Processes all synthetic titles in parallel batches of 5

BATCH_SIZE = 5
total_records = len(df)

def process_row(row):
    title = row['CRM_OCCUPATION_TITLE']
    cleaned = row['cleaned_title']
    rule_label = row['fast_path'] if pd.notna(row['fast_path']) else None

    llm_result = classify_title(cleaned)

    if rule_label is None:
        validation = 'NEEDS_REVIEW' if llm_result['review_status'] == 'NEEDS_REVIEW' else 'LLM_ONLY'
    elif rule_label == llm_result['classification']:
        validation = 'AGREE'
    else:
        validation = 'DISAGREE'

    if llm_result['confidence'] < CONFIDENCE_THRESHOLD:
        validation = 'NEEDS_REVIEW'

    return {
        'original_title': title,
        'rule_based_label': rule_label if rule_label else 'NONE',
        'llm_label': llm_result['classification'],
        'llm_major_group': llm_result['major_group'],
        'llm_ssoc_code': llm_result['ssoc_code'],
        'llm_ssoc_occupation_title': llm_result['ssoc_occupation_title'],
        'llm_confidence': llm_result['confidence'],
        'llm_votes': llm_result['votes'],
        'llm_reasoning': llm_result['reasoning'],
        'validation_status': validation,
        'prompt_version': llm_result['prompt_version']
    }

print(f"Running classification on {total_records} titles...")
print(f"Model: {MODEL} | Prompt: {PROMPT_VERSION} | Batch size: {BATCH_SIZE}\n")

results_list = []
rows = [row for _, row in df.iterrows()]
start = time.time()

for batch_start in range(0, len(rows), BATCH_SIZE):
    batch = rows[batch_start:batch_start + BATCH_SIZE]

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as executor:
        futures = {executor.submit(process_row, row): row for row in batch}
        for future in as_completed(futures):
            try:
                results_list.append(future.result())
            except Exception as e:
                print(f"Error: {e}")

    completed = min(batch_start + BATCH_SIZE, len(rows))
    elapsed = time.time() - start
    rate = completed / elapsed
    eta = (len(rows) - completed) / rate if rate > 0 else 0
    print(f"  {completed}/{total_records} | Elapsed: {elapsed:.0f}s | ETA: {eta:.0f}s")

total_time = time.time() - start
results_df = pd.DataFrame(results_list)

print(f"\nDone in {total_time:.1f}s ({total_time/60:.1f} mins)")
print(f"Avg per record: {total_time/total_records:.1f}s")
print(f"\nLabel distribution: {results_df['llm_label'].value_counts().to_dict()}")
print(f"Validation status:  {results_df['validation_status'].value_counts().to_dict()}")
print(f"Avg confidence:     {results_df['llm_confidence'].mean():.3f}")

In [ ]:
results_df.to_csv("data/synthetic_results.csv", index=False)
print(f"Saved {len(results_df)} records to data/synthetic_results.csv")

In [ ]:
# Show the disagreements and needs review
print("=" * 70)
print("DISAGREE — rule-based and LLM diverged")
print("=" * 70)
print(results_df[results_df['validation_status'] == 'DISAGREE'][
    ['original_title', 'rule_based_label', 'llm_label',
     'llm_confidence', 'llm_reasoning']
].to_string(index=False))

print("\n")
print("=" * 70)
print("NEEDS REVIEW — low confidence or ambiguous")
print("=" * 70)
print(results_df[results_df['validation_status'] == 'NEEDS_REVIEW'][
    ['original_title', 'llm_label', 'llm_confidence', 'llm_reasoning']
].to_string(index=False))

enter this in the terminal to enter streamlit:

source .venv/bin/activate
streamlit run app/review_app.py

In [ ]:
# ── Merge LLM results with human review decisions ─────────────────────────────
# Human review decisions stored in data/reviewed_labels.csv (gitignored)
# Run the Streamlit review app first: streamlit run app/review_app.py

reviewed = pd.read_csv("data/reviewed_labels.csv")
results = pd.read_csv("data/synthetic_results.csv")
results['llm_label'] = results['llm_label'].fillna('NA')

final = results.merge(
    reviewed[['original_title', 'final_label', 'reviewed_by', 'reviewed_at']],
    on='original_title',
    how='left'
)

final['final_label'] = final['final_label'].fillna(final['llm_label'])
final['label_source'] = final.apply(
    lambda r: 'HUMAN' if r['original_title'] in reviewed['original_title'].values
    else 'LLM', axis=1
)

final.to_csv("data/final_labels.csv", index=False)

print(f"Total records: {len(final)}")
print(f"\nFinal label distribution:")
print(final['final_label'].value_counts())
print(f"\nLabel source:")
print(final['label_source'].value_counts())